In [19]:
from pathlib import Path

import polars as pl
import polars_h3 as plh3
import yaml


In [35]:
def process(
    data: pl.DataFrame,
    res: int,
    work_hours_between: tuple[int, int] = (9, 16),
    home_hours_between: tuple[int, int] = (20, 7),
) -> pl.DataFrame:
    temp = data.with_columns(
        pl.col.interval_start.dt.date().alias("day"),
        pl.col.interval_start.dt.time().alias("time"),
        pl.col.interval_start.dt.hour().alias("hour"),
        pl.col.interval_start.dt.weekday().alias("dow"),
        plh3.int_to_str(
            plh3.latlng_to_cell(pl.col("latitude_mean"), pl.col("longitude_mean"), res)
        ).alias(f"h3_{res}"),
    ).with_columns(
        pl.when(
            (pl.col.dow.is_between(1, 5))
            & (pl.col.hour.is_between(*work_hours_between))
        )
        .then(pl.lit("work"))
        .otherwise(
            pl.when(
                (pl.col.dow.is_between(1, 5))
                & (
                    (pl.col.hour <= home_hours_between[1])
                    | (pl.col.hour >= home_hours_between[0])
                )
                | (pl.col.dow > 5)
            )
            .then(pl.lit("home"))
            .otherwise(pl.lit("other"))
        )
        .alias("label")
    )
    temp = temp.join(hw, on="device_id")

    hn = (
        temp.filter(
            (pl.col(f"h3_{res}") == pl.col(f"h3_{res}_home")) & (pl.col.label == "home")
        )
        .group_by(["device_id", "day"])
        .agg(pl.col(f"h3_{res}").count().alias("home_count"))
        .sort(["device_id", "day"])
    )
    wn = (
        temp.filter(
            (pl.col(f"h3_{res}") == pl.col(f"h3_{res}_work")) & (pl.col.label == "work")
        )
        .group_by(["device_id", "day"])
        .agg(pl.col(f"h3_{res}").count().alias("work_count"))
        .sort(["device_id", "day"])
    )
    m = pl.concat(
        [
            hn.join(wn, on=["device_id", "day"], how="left")
            .fill_null(0)
            .sort(["device_id", "day"]),
            hn.join(wn, on=["device_id", "day"], how="right")
            .fill_null(0)
            .select("device_id", "day", "home_count", "work_count")
            .sort(["device_id", "day"]),
        ]
    ).sort(["device_id", "day"])
    return m


def merge_with_hw(
    data: pl.DataFrame,
    hw: pl.DataFrame,
    home_threshold: int = 0,
    work_threshold: int = 0,
) -> pl.DataFrame:
    return (
        data.join(hw, on="device_id")
        .filter(
            (pl.col.count_home >= home_threshold)
            & (pl.col.count_work >= work_threshold)
        )
        .rename({"home_count": "stops_at_home", "work_count": "stops_at_work"})
    )

In [3]:
with open("../config.budapest.yaml") as fp:
    config = yaml.safe_load(fp)

In [36]:
DIR = f"{config['dirs']['output']}/{config['city']}"
RES = 8
HOME_THRESHOLD = 5
WORK_THRESHOLD = 5

In [37]:
hw = pl.read_csv(f"{DIR}/homes/v2/home_work_h3_{RES}.csv")
hw.head(3)

device_id,h3_8_home,count_home,h3_8_work,count_work
i64,str,i64,str,i64
1,"""881e037857fffff""",53,"""881e037857fffff""",35
2,"""881e0378c1fffff""",4,"""881e037833fffff""",22
3,"""881e1d8091fffff""",245,"""881e1d8359fffff""",143


In [38]:
columns = [
    "device_id",
    "interval_start",
    "latitude_mean",
    "longitude_mean",
    "interval_lenght",
]
Path(f"{DIR}/dynamic_commuting/{RES}").mkdir(parents=True, exist_ok=True)
result = pl.DataFrame()
for i in range(256):
    data = pl.read_parquet(f"{DIR}/stops/v2/{i:02x}.parquet", columns=columns)
    m = process(data, RES)  # runs 45s
    result = pl.concat([result, m])
result.write_csv(f"{DIR}/dynamic_commuting/{RES}/home_work_stops_by_daily.csv")

merged = merge_with_hw(result, hw, HOME_THRESHOLD, WORK_THRESHOLD)
merged.write_csv(f"{DIR}/dynamic_commuting/{RES}/number_of_stops_at_home_work.csv")